# 第 39 课｜合并行情与财报：point-in-time 数据

本 Notebook 对应「365 天量化金融」第 39 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里将课程 API 重建为可离线执行、可核对的量化教学实现，并补充数据契约、断言与方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/39_youtube-iksBIxLykTU/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情与本课内构造的模拟财报（**均非真实证券数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **来源边界**：课程材料没有提供可独立核验的完整 Python 源码和真实行情快照。本 Notebook 复现课程方法，不把口播中的股票、数值或图形当作离线验收基准。

## 1. 本节量化问题

行情通常是高频长表，行业是低频维表，财报又按季度发布。研究任务不是“让两张表成功拼起来”，而是让每个交易时点只看到当时已经公开且可以使用的信息。

本节按由小到大的顺序回答四个问题：

1. `pd.concat` 沿行、沿列分别改变什么；
2. `merge` 的 `inner/outer/left` 如何决定保留哪些精确键；
3. `validate` 和 `indicator` 如何把多对多膨胀、未匹配键变成可检查结果；
4. `merge_asof(direction="backward")` 如何构造 point-in-time 财报快照，并证明使用 `report_period` 会造成未来数据泄漏。

核心原则：

> **财报属于哪个季度，不等于市场何时知道这份财报。**

## 2. 输入、输出与验收

**输入**

- `daily_prices.csv`：固定种子生成的 6 只虚构资产日线；
- 本 Notebook 内构造的模拟行业维表和模拟财报发布表；
- 所有 ROE 均为十进制的累计年初至今口径，例如 `0.08 = 8%`。

**输出**

- 行、列拼接的带标签小表；
- 精确键连接的匹配审计；
- 每个 `(trade_at, symbol)` 对应的最近已知财报；
- 发布前后可直接核对的阶梯式 ROE 快照。

**验收**

- 行情联合键唯一，行业维表每个 `symbol` 至多一行；
- point-in-time 合并不改变行情行数；
- 每个已匹配的 `available_at <= trade_at`；
- 公告在收盘后发布时，当日 15:00 快照看不到，下一可用时点才看到；
- 原始 ROE 排名只作为表连接演示，不冒充策略回测。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 3. `concat(axis=0)`：沿行追加同结构记录

先用两只虚构资产的确定性小表观察默认行为。两张表字段相同，`axis=0` 沿行轴追加：

- 原来每张表 2 行，拼接后应有 4 行；
- `ignore_index=True` 丢弃各自的旧行号，重新生成 `0..3`；
- 它不检查业务主键，真实数据拼接后仍要检查 `(date, symbol)` 是否重复。

In [2]:
# 两张表字段完全相同；每一行仍代表“某日、某只资产的一条价格记录”。
alpha_rows = pd.DataFrame(
    {
        "date": pd.to_datetime(["2025-01-02", "2025-01-03"]),
        "symbol": ["ALPHA", "ALPHA"],
        "close": [100.0, 101.0],  # ALPHA 两日收盘价
    }
)
beta_rows = pd.DataFrame(
    {
        "date": pd.to_datetime(["2025-01-02", "2025-01-03"]),
        "symbol": ["BETA", "BETA"],
        "close": [50.0, 49.5],    # BETA 两日收盘价
    }
)

# axis=0 沿行追加；ignore_index=True 重新生成连续行号。
row_concat = pd.concat(
    [alpha_rows, beta_rows],
    axis=0,
    ignore_index=True,
)

assert row_concat.shape == (4, 3)
assert row_concat.index.tolist() == [0, 1, 2, 3]
assert not row_concat.duplicated(["date", "symbol"]).any()
print("沿行拼接后 shape =", row_concat.shape, "含义：4 条价格记录 × 3 个字段")
row_concat

沿行拼接后 shape = (4, 3) 含义：4 条价格记录 × 3 个字段


,date,symbol,close
0,2025-01-02,ALPHA,100.0
1,2025-01-03,ALPHA,101.0
2,2025-01-02,BETA,50.0
3,2025-01-03,BETA,49.5


## 4. `concat(axis=1)`：按标签并排放置列

沿列拼接不是按“第几行”硬贴，而是先对齐索引标签。下面两只资产的日期范围错开一天：

- 结果索引取两个日期集合的并集，共 4 天；
- `ALPHA` 在 1 月 6 日没有记录，`BETA` 在 1 月 2 日没有记录；
- 两个缺口显示为 `NaN`，不能在没有业务依据时直接填成 0。

In [3]:
# 两个 Series 的日期标签并不完全相同。
alpha_close = pd.Series(
    [100.0, 101.0, 102.0],
    index=pd.to_datetime(["2025-01-02", "2025-01-03", "2025-01-06"]),
    name="ALPHA",
)
beta_close = pd.Series(
    [49.5, 50.5, 51.0],
    index=pd.to_datetime(["2025-01-03", "2025-01-06", "2025-01-07"]),
    name="BETA",
)

# axis=1 新增列；Pandas 先按日期标签并集对齐。
column_concat = pd.concat(
    [alpha_close, beta_close],
    axis=1,
)

assert column_concat.shape == (4, 2)
assert column_concat.isna().sum().to_dict() == {"ALPHA": 1, "BETA": 1}
print("沿列拼接后 shape =", column_concat.shape, "含义：日期并集 × 两只资产")
column_concat

沿列拼接后 shape = (4, 2) 含义：日期并集 × 两只资产


,ALPHA,BETA
2025-01-02,100.0,NaN
2025-01-03,101.0,49.5
2025-01-06,102.0,50.5
2025-01-07,NaN,51.0


## 5. `merge`：用精确业务键连接事实表和维表

`merge` 与 `concat` 的关键差别是显式声明连接键。下面价格表有 `ALPHA/BETA/GAMMA`，行业维表有 `ALPHA/BETA/DELTA`。

| `how` | 保留的键 | 本例用途 |
|---|---|---|
| `inner` | 两边交集 | 只看两表都存在的资产 |
| `left` | 左表全部键 | 给行情补充行业且不丢行情 |
| `outer` | 两边并集 | 审计双方各自缺失的键 |

`indicator=True` 生成 `_merge`：`both/left_only/right_only` 分别表示匹配、只在左表、只在右表。它比只看 `NaN` 更准确，因为原始字段本身也可能合法缺失。

In [4]:
# 价格事实表包含 3 只资产；行业维表故意缺 GAMMA、额外含 DELTA。
price_today = pd.DataFrame(
    {
        "symbol": ["ALPHA", "BETA", "GAMMA"],
        "close": [101.0, 49.5, 80.0],
    }
)
sector_dimension = pd.DataFrame(
    {
        "symbol": ["ALPHA", "BETA", "DELTA"],
        "sector": ["科技", "金融", "公用事业"],
    }
)

# validate="many_to_one" 声明：左侧可有多条行情，右侧每个 symbol 只能有一行。
inner_join = price_today.merge(
    sector_dimension,
    on="symbol",
    how="inner",
    validate="many_to_one",
)
left_join = price_today.merge(
    sector_dimension,
    on="symbol",
    how="left",
    validate="many_to_one",
    indicator=True,
)
outer_join = price_today.merge(
    sector_dimension,
    on="symbol",
    how="outer",
    validate="many_to_one",
    indicator=True,
)

assert set(inner_join["symbol"]) == {"ALPHA", "BETA"}
assert len(left_join) == len(price_today)
assert left_join["_merge"].eq("left_only").sum() == 1
assert set(outer_join.loc[outer_join["_merge"].eq("right_only"), "symbol"]) == {"DELTA"}

print("inner 行数 =", len(inner_join), "；left 行数 =", len(left_join), "；outer 行数 =", len(outer_join))
outer_join.sort_values("symbol").reset_index(drop=True)

inner 行数 = 2 ；left 行数 = 3 ；outer 行数 = 4


,symbol,close,sector,_merge
0,ALPHA,101.0,科技,both
1,BETA,49.5,金融,both
2,DELTA,NaN,公用事业,right_only
3,GAMMA,80.0,NaN,left_only


## 6. `validate`：在结果膨胀前拦截错误基数

如果行业维表中同一资产出现两行，普通 `merge` 会复制左表记录。代码仍能运行，但金额、收益和样本数都可能被重复计算。

下面故意制造重复维度键，并在 `try/except` 中捕获预期的 `MergeError`。Notebook 不产生红色报错，同时明确展示保护机制确实生效。

In [5]:
# ALPHA 在右表出现两次，不再满足 many_to_one。
duplicated_dimension = pd.concat(
    [sector_dimension, sector_dimension.iloc[[0]]],
    ignore_index=True,
)

try:
    price_today.merge(
        duplicated_dimension,
        on="symbol",
        how="left",
        validate="many_to_one",
    )
except pd.errors.MergeError as error:
    print("预期拦截：行业维表的 symbol 不唯一。")
    print("异常类型：", type(error).__name__)
else:
    raise AssertionError("validate 未拦截重复维度键")

assert duplicated_dimension["symbol"].duplicated().any()

预期拦截：行业维表的 symbol 不唯一。
异常类型： MergeError


## 7. point-in-time 的三个时间字段

财报连接至少要区分：

| 字段 | 含义 | 能否直接作为交易时点连接键 |
|---|---|---|
| `report_period` | 财报覆盖期末，例如 2024-12-31 | **不能**；此时报告通常尚未发布 |
| `published_at` | 发布源记录的公告时间 | 还要结合时区、收盘和处理延迟 |
| `available_at` | 研究系统确认可使用的最早时点 | 可以；但生成规则必须可追溯 |

本小例子把交易快照固定在 15:00。`ALPHA` 的年报在 1 月 3 日 18:00 发布，设定为 1 月 6 日 09:30 可用；因此 1 月 3 日收盘快照只能看到旧报告。

所有名称、价格、ROE 和时间均为教学模拟数据。

In [6]:
# 交易事实表：3 个交易时点 × 2 只虚构资产。
trade_demo = pd.DataFrame(
    {
        "trade_at": pd.to_datetime(
            [
                "2025-01-03 15:00", "2025-01-03 15:00",
                "2025-01-06 15:00", "2025-01-06 15:00",
                "2025-01-08 15:00", "2025-01-08 15:00",
            ]
        ),
        "symbol": ["ALPHA", "BETA", "ALPHA", "BETA", "ALPHA", "BETA"],
        "close": [101.0, 49.5, 102.0, 50.5, 103.0, 51.0],
    }
)

# 财报表：roe_ytd 是十进制累计 ROE；0.10 表示 10%。
fundamental_demo = pd.DataFrame(
    {
        "symbol": ["ALPHA", "BETA", "ALPHA", "BETA"],
        "report_period": pd.to_datetime(
            ["2024-09-30", "2024-09-30", "2024-12-31", "2024-12-31"]
        ),
        "published_at": pd.to_datetime(
            [
                "2024-11-01 18:00",
                "2024-11-01 18:00",
                "2025-01-03 18:00",  # 当日 15:00 之后发布
                "2025-01-07 18:00",
            ]
        ),
        "available_at": pd.to_datetime(
            [
                "2024-11-04 09:30",
                "2024-11-04 09:30",
                "2025-01-06 09:30",  # ALPHA 新报告从这里开始可见
                "2025-01-08 09:30",  # BETA 新报告从这里开始可见
            ]
        ),
        "roe_ytd": [0.10, 0.08, 0.12, 0.09],
    }
)

assert not trade_demo.duplicated(["trade_at", "symbol"]).any()
assert not fundamental_demo.duplicated(["available_at", "symbol"]).any()
assert fundamental_demo["published_at"].le(fundamental_demo["available_at"]).all()
fundamental_demo

,symbol,report_period,published_at,available_at,roe_ytd
0,ALPHA,2024-09-30,2024-11-01 18:00:00,2024-11-04 09:30:00,0.10
1,BETA,2024-09-30,2024-11-01 18:00:00,2024-11-04 09:30:00,0.08
2,ALPHA,2024-12-31,2025-01-03 18:00:00,2025-01-06 09:30:00,0.12
3,BETA,2024-12-31,2025-01-07 18:00:00,2025-01-08 09:30:00,0.09


## 8. `merge_asof`：只向后找最近已知记录

关键参数逐一对应业务约束：

- `left_on="trade_at"`：每个行情快照的决策时点；
- `right_on="available_at"`：财报真正可使用的时点；
- `by="symbol"`：只允许同一资产内部匹配；
- `direction="backward"`：只找不晚于交易时点的最近记录；
- `tolerance=180D`：超过 180 天的财报视为陈旧并返回缺失；
- `allow_exact_matches=True`：恰好在可用时点的记录可以使用。

`merge_asof` 要求连接时间全局升序，因此这里先按“时间、资产”排序。若先按资产分组再让时间从头开始，许多 Pandas 版本会拒绝执行。

In [7]:
# merge_asof 的 on 键必须全局升序；symbol 作为同一时点的稳定次序。
trade_demo_sorted = trade_demo.sort_values(
    ["trade_at", "symbol"]
).reset_index(drop=True)
fundamental_demo_sorted = fundamental_demo.sort_values(
    ["available_at", "symbol"]
).reset_index(drop=True)

pit_demo = pd.merge_asof(
    trade_demo_sorted,
    fundamental_demo_sorted,
    left_on="trade_at",
    right_on="available_at",
    by="symbol",
    direction="backward",
    tolerance=pd.Timedelta("180D"),
    allow_exact_matches=True,
)

# 预期阶梯：
# ALPHA: 10% -> 12% -> 12%
# BETA :  8% ->  8% ->  9%
observed_roe = (
    pit_demo.pivot(
        index="trade_at",
        columns="symbol",
        values="roe_ytd",
    )
    .sort_index()
    [["ALPHA", "BETA"]]
)
expected_roe = np.array(
    [
        [0.10, 0.08],
        [0.12, 0.08],
        [0.12, 0.09],
    ]
)

np.testing.assert_allclose(observed_roe.to_numpy(), expected_roe)
assert len(pit_demo) == len(trade_demo)
assert pit_demo["available_at"].le(pit_demo["trade_at"]).all()
print("point-in-time 校验通过：发布前看旧值，available_at 到达后才切换。")
observed_roe.style.format("{:.1%}")

point-in-time 校验通过：发布前看旧值，available_at 到达后才切换。


symbol,ALPHA,BETA
trade_at,,
2025-01-03 15:00:00,10.0%,8.0%
2025-01-06 15:00:00,12.0%,8.0%
2025-01-08 15:00:00,12.0%,9.0%


## 9. 反例：用 `report_period` 会把未来财报带回过去

下面只替换右侧时间键，错误地按财报期末向后找。2024 年报的 `report_period=2024-12-31`，所以 2025-01-03 的交易快照会匹配到尚未发布的年报。

这正是 look-ahead bias：数值、shape、排序都可能看起来正常，但当时的市场参与者并不知道这份数据。

In [8]:
# 错误示范：把财报覆盖期末当成市场可知时点。
leaky_demo = pd.merge_asof(
    trade_demo.sort_values(["trade_at", "symbol"]),
    fundamental_demo.sort_values(["report_period", "symbol"]),
    left_on="trade_at",
    right_on="report_period",
    by="symbol",
    direction="backward",
    tolerance=pd.Timedelta("180D"),
)

leak_audit = (
    pit_demo[["trade_at", "symbol", "roe_ytd"]]
    .rename(columns={"roe_ytd": "正确：按 available_at"})
    .merge(
        leaky_demo[["trade_at", "symbol", "roe_ytd"]].rename(
            columns={"roe_ytd": "错误：按 report_period"}
        ),
        on=["trade_at", "symbol"],
        how="inner",
        validate="one_to_one",
    )
)
leak_audit["是否泄漏"] = (
    leak_audit["正确：按 available_at"]
    != leak_audit["错误：按 report_period"]
)

alpha_jan3 = leak_audit.loc[
    leak_audit["trade_at"].eq(pd.Timestamp("2025-01-03 15:00"))
    & leak_audit["symbol"].eq("ALPHA")
].iloc[0]
assert np.isclose(alpha_jan3["正确：按 available_at"], 0.10)
assert np.isclose(alpha_jan3["错误：按 report_period"], 0.12)
assert leak_audit["是否泄漏"].any()
print("已证明：仅把连接键换成 report_period，就会把未发布数据带回过去。")
leak_audit

已证明：仅把连接键换成 report_period，就会把未发布数据带回过去。


,trade_at,symbol,正确：按 available_at,错误：按 report_period,是否泄漏
0,2025-01-03 15:00:00,ALPHA,0.10,0.12,True
1,2025-01-03 15:00:00,BETA,0.08,0.09,True
2,2025-01-06 15:00:00,ALPHA,0.12,0.12,False
3,2025-01-06 15:00:00,BETA,0.08,0.09,True
4,2025-01-08 15:00:00,ALPHA,0.12,0.12,False
5,2025-01-08 15:00:00,BETA,0.09,0.09,False


## 10. `tolerance`：让陈旧财报显式失效

`direction="backward"` 只保证“不看未来”，并不限制向过去追溯多久。若最后一份财报已经非常陈旧，继续沿用也可能不符合研究定义。

下面把交易时点推进到 2025 年 10 月；最近财报已超过 180 天，因此 `roe_ytd` 应为 `NaN`。这里的缺失表示“没有满足时效规则的记录”，不能擅自解释成 ROE 为 0。

In [9]:
# 单个远期快照用于验证 tolerance；价格本身只作占位。
stale_trade = pd.DataFrame(
    {
        "trade_at": [pd.Timestamp("2025-10-01 15:00")],
        "symbol": ["ALPHA"],
        "close": [110.0],
    }
)
stale_result = pd.merge_asof(
    stale_trade.sort_values(["trade_at", "symbol"]),
    fundamental_demo.sort_values(["available_at", "symbol"]),
    left_on="trade_at",
    right_on="available_at",
    by="symbol",
    direction="backward",
    tolerance=pd.Timedelta("180D"),
)

assert pd.isna(stale_result.loc[0, "roe_ytd"])
print("陈旧记录已拒绝：超过 180 天后 roe_ytd =", stale_result.loc[0, "roe_ytd"])
stale_result

陈旧记录已拒绝：超过 180 天后 roe_ytd = nan


,trade_at,symbol,close,report_period,published_at,available_at,roe_ytd
0,2025-10-01 15:00:00,ALPHA,110.0,NaT,NaT,NaT,NaN


## 11. 回到完整模拟行情：建立行情事实表

下面读取 6 只虚构资产，并截取 2024-03-20 至 2024-09-06：

- `date` 是交易日期标签；
- `trade_at = date + 15:00` 是本课约定的收盘研究快照；
- `(trade_at, symbol)` 必须唯一；
- 这里的“工作日”来自 fixture，不等于真实交易所日历。

先验证原始行情，再连接任何维度或财报。

In [10]:
# 读取固定种子的模拟日线；只保留本课所需时段和字段。
daily = pd.read_csv(
    DATA_DIR / "daily_prices.csv",
    parse_dates=["date"],
)
market = (
    daily.loc[
        daily["date"].between("2024-03-20", "2024-09-06"),
        ["date", "symbol", "adj_close", "volume"],
    ]
    .sort_values(["date", "symbol"])
    .reset_index(drop=True)
)
market["trade_at"] = market["date"] + pd.Timedelta(hours=15)

assert market["symbol"].nunique() == 6
assert not market.duplicated(["trade_at", "symbol"]).any()
assert market["adj_close"].gt(0).all()
assert market["volume"].ge(0).all()
print("行情 shape =", market.shape)
print("日期范围 =", market["date"].min().date(), "至", market["date"].max().date())
print("资产 =", market["symbol"].drop_duplicates().tolist())
market.head(6)

行情 shape = (738, 5)
日期范围 = 2024-03-20 至 2024-09-06
资产 = ['BANK_A', 'ENERGY_A', 'EV_A', 'INSURE_A', 'POWER_A', 'TECH_A']


,date,symbol,adj_close,volume,trade_at
0,2024-03-20,BANK_A,11.5419,1940096,2024-03-20 15:00:00
1,2024-03-20,ENERGY_A,7.0203,6493192,2024-03-20 15:00:00
2,2024-03-20,EV_A,27.7555,5228792,2024-03-20 15:00:00
3,2024-03-20,INSURE_A,36.4239,4642774,2024-03-20 15:00:00
4,2024-03-20,POWER_A,7.8246,3235720,2024-03-20 15:00:00
5,2024-03-20,TECH_A,50.9082,3896559,2024-03-20 15:00:00


## 12. 先用精确键补行业

行业是每只资产一条记录的维表，适合普通 `merge`：

- 从原始长表提取唯一 `(symbol, sector)`；
- 合并前断言 `symbol` 唯一；
- 用 `how="left"` 保留全部行情；
- 用 `validate="many_to_one"` 声明右表基数；
- 用 `_merge` 验证本样本没有行业缺口。

真实历史行业分类可能随时间变化；那时它本身也是 point-in-time 维度，不能压成静态一行。

In [11]:
# 构造本 fixture 的静态行业维表，并给每条行情补充行业。
sector_dimension_full = (
    daily[["symbol", "sector"]]
    .drop_duplicates()
    .sort_values("symbol")
    .reset_index(drop=True)
)
assert not sector_dimension_full["symbol"].duplicated().any()

market_with_sector = market.merge(
    sector_dimension_full,
    on="symbol",
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert len(market_with_sector) == len(market)
assert market_with_sector["_merge"].eq("both").all()
print("行业匹配率 =", f"{market_with_sector['_merge'].eq('both').mean():.1%}")
market_with_sector.head(6)

行业匹配率 = 100.0%


,date,symbol,adj_close,volume,trade_at,sector,_merge
0,2024-03-20,BANK_A,11.5419,1940096,2024-03-20 15:00:00,金融,both
1,2024-03-20,ENERGY_A,7.0203,6493192,2024-03-20 15:00:00,能源,both
2,2024-03-20,EV_A,27.7555,5228792,2024-03-20 15:00:00,新能源,both
3,2024-03-20,INSURE_A,36.4239,4642774,2024-03-20 15:00:00,金融,both
4,2024-03-20,POWER_A,7.8246,3235720,2024-03-20 15:00:00,公用事业,both
5,2024-03-20,TECH_A,50.9082,3896559,2024-03-20 15:00:00,科技,both


## 13. 构造可追溯的模拟财报发布表

为每只虚构资产建立 4 期累计 ROE：

- `annual_roe_reference` 只是生成数值的教学参数，不是真实公司 ROE；
- `ytd_fraction` 让 Q1、半年、前三季度、全年呈现累计口径；
- `published_at` 是模拟发布时刻；
- `available_at` 是本研究系统允许使用的时刻；
- 同一 `(symbol, available_at)` 只能有一条记录。

注意：并非所有数据库中的 ROE 都是累计口径。季度单季、年化、TTM 和平均净资产口径不能混用，必须读取字段定义。

In [12]:
# 每只虚构资产的教学年化参照值；0.16 表示 16%。
annual_roe_table = pd.DataFrame(
    {
        "symbol": ["BANK_A", "ENERGY_A", "EV_A", "INSURE_A", "POWER_A", "TECH_A"],
        "annual_roe_reference": [0.10, 0.11, 0.14, 0.09, 0.07, 0.16],
    }
)

# 四期报告的覆盖期、模拟发布时间、可用时点和累计比例。
release_schedule = pd.DataFrame(
    {
        "report_label": ["2023Q3", "2023FY", "2024Q1", "2024H1"],
        "report_period": pd.to_datetime(
            ["2023-09-30", "2023-12-31", "2024-03-31", "2024-06-30"]
        ),
        "published_at": pd.to_datetime(
            [
                "2023-10-30 18:00",
                "2024-03-28 18:00",
                "2024-04-26 18:00",
                "2024-08-29 18:00",
            ]
        ),
        "available_at": pd.to_datetime(
            [
                "2023-10-31 09:30",
                "2024-03-29 09:30",
                "2024-04-29 09:30",
                "2024-08-30 09:30",
            ]
        ),
        "ytd_fraction": [0.75, 1.00, 0.25, 0.50],
    }
)

# 这里有意做笛卡尔积：6 只资产 × 4 个报告期 = 24 条模拟财报。
fundamentals = annual_roe_table.merge(
    release_schedule,
    how="cross",
)
fundamentals["roe_ytd"] = (
    fundamentals["annual_roe_reference"]
    * fundamentals["ytd_fraction"]
).round(4)
fundamentals = fundamentals[
    [
        "symbol",
        "report_label",
        "report_period",
        "published_at",
        "available_at",
        "roe_ytd",
    ]
].sort_values(["available_at", "symbol"]).reset_index(drop=True)

assert fundamentals.shape == (24, 6)
assert not fundamentals.duplicated(["symbol", "available_at"]).any()
assert fundamentals["published_at"].le(fundamentals["available_at"]).all()
print("模拟财报 shape =", fundamentals.shape, "含义：资产 × 报告期")
fundamentals.head(8)

模拟财报 shape = (24, 6) 含义：资产 × 报告期


,symbol,report_label,report_period,published_at,available_at,roe_ytd
0,BANK_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0750
1,ENERGY_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0825
2,EV_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.1050
3,INSURE_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0675
4,POWER_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0525
5,TECH_A,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.1200
6,BANK_A,2023FY,2023-12-31,2024-03-28 18:00:00,2024-03-29 09:30:00,0.1000
7,ENERGY_A,2023FY,2023-12-31,2024-03-28 18:00:00,2024-03-29 09:30:00,0.1100


## 14. 完整 point-in-time 合并

行业补充使用精确键，财报补充使用时点键。这里设置 220 天时效窗口，是为了让样本开始时仍可使用 2023Q3；它只是本教学例子的研究规则，不是通用财报保鲜期。

验收不只检查“没有报错”，还检查：

1. 合并前后行情行数一致；
2. 行情联合键仍唯一；
3. 所有匹配记录的 `available_at` 不晚于 `trade_at`；
4. 本确定性样本每个时点都能找到 220 天内的报告。

In [13]:
# 删除 merge 审计列后，按时间、资产排序并执行 point-in-time 合并。
market_for_pit = (
    market_with_sector.drop(columns="_merge")
    .sort_values(["trade_at", "symbol"])
    .reset_index(drop=True)
)
fundamentals_for_pit = (
    fundamentals.sort_values(["available_at", "symbol"])
    .reset_index(drop=True)
)

pit_result = pd.merge_asof(
    market_for_pit,
    fundamentals_for_pit,
    left_on="trade_at",
    right_on="available_at",
    by="symbol",
    direction="backward",
    tolerance=pd.Timedelta("220D"),
    allow_exact_matches=True,
)

matched = pit_result["available_at"].notna()
assert len(pit_result) == len(market_for_pit)
assert not pit_result.duplicated(["trade_at", "symbol"]).any()
assert (
    pit_result.loc[matched, "available_at"]
    <= pit_result.loc[matched, "trade_at"]
).all()
assert matched.all()

print("PIT 结果 shape =", pit_result.shape)
print("行情保留率 =", f"{len(pit_result) / len(market_for_pit):.1%}")
print("220 天内财报匹配率 =", f"{matched.mean():.1%}")
pit_result.head(6)

PIT 结果 shape = (738, 11)
行情保留率 = 100.0%
220 天内财报匹配率 = 100.0%


,date,symbol,adj_close,volume,trade_at,sector,report_label,report_period,published_at,available_at,roe_ytd
0,2024-03-20,BANK_A,11.5419,1940096,2024-03-20 15:00:00,金融,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0750
1,2024-03-20,ENERGY_A,7.0203,6493192,2024-03-20 15:00:00,能源,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0825
2,2024-03-20,EV_A,27.7555,5228792,2024-03-20 15:00:00,新能源,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.1050
3,2024-03-20,INSURE_A,36.4239,4642774,2024-03-20 15:00:00,金融,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0675
4,2024-03-20,POWER_A,7.8246,3235720,2024-03-20 15:00:00,公用事业,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.0525
5,2024-03-20,TECH_A,50.9082,3896559,2024-03-20 15:00:00,科技,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.1200


## 15. 发布日前后审计：结果应呈阶梯变化

选 `TECH_A` 在三个发布窗口前后核对：

- 2024-03-28 15:00：年报 18:00 才发布，仍应看到 2023Q3；
- 2024-03-29 15:00：年报已在 09:30 可用，应切到 2023FY；
- 2024-04-26 15:00：Q1 尚未发布，仍看到 2023FY；
- 2024-04-29 15:00：切到 2024Q1；
- 2024-08-29 15:00：半年报尚未发布，仍看到 2024Q1；
- 2024-08-30 15:00：切到 2024H1。

这种“跨发布日期才跳变”的阶梯结构，是 point-in-time 快照最直观的检查之一。

In [14]:
# 抽取 TECH_A 的发布窗口，并把 ROE 额外显示为百分数。
audit_dates = pd.to_datetime(
    [
        "2024-03-28",
        "2024-03-29",
        "2024-04-26",
        "2024-04-29",
        "2024-08-29",
        "2024-08-30",
    ]
)
tech_audit = (
    pit_result.loc[
        pit_result["symbol"].eq("TECH_A")
        & pit_result["date"].isin(audit_dates),
        [
            "date",
            "trade_at",
            "report_label",
            "report_period",
            "published_at",
            "available_at",
            "roe_ytd",
        ],
    ]
    .sort_values("trade_at")
    .assign(roe_ytd_percent=lambda frame: frame["roe_ytd"] * 100)
)

reported_period_by_date = tech_audit.set_index("date")["report_period"]
assert reported_period_by_date.loc[pd.Timestamp("2024-03-28")] == pd.Timestamp("2023-09-30")
assert reported_period_by_date.loc[pd.Timestamp("2024-03-29")] == pd.Timestamp("2023-12-31")
assert reported_period_by_date.loc[pd.Timestamp("2024-04-26")] == pd.Timestamp("2023-12-31")
assert reported_period_by_date.loc[pd.Timestamp("2024-04-29")] == pd.Timestamp("2024-03-31")
assert reported_period_by_date.loc[pd.Timestamp("2024-08-29")] == pd.Timestamp("2024-03-31")
assert reported_period_by_date.loc[pd.Timestamp("2024-08-30")] == pd.Timestamp("2024-06-30")
print("发布窗口断言全部通过。")
tech_audit

发布窗口断言全部通过。


,date,trade_at,report_label,report_period,published_at,available_at,roe_ytd,roe_ytd_percent
41,2024-03-28,2024-03-28 15:00:00,2023Q3,2023-09-30,2023-10-30 18:00:00,2023-10-31 09:30:00,0.12,12.0
47,2024-03-29,2024-03-29 15:00:00,2023FY,2023-12-31,2024-03-28 18:00:00,2024-03-29 09:30:00,0.16,16.0
167,2024-04-26,2024-04-26 15:00:00,2023FY,2023-12-31,2024-03-28 18:00:00,2024-03-29 09:30:00,0.16,16.0
173,2024-04-29,2024-04-29 15:00:00,2024Q1,2024-03-31,2024-04-26 18:00:00,2024-04-29 09:30:00,0.04,4.0
701,2024-08-29,2024-08-29 15:00:00,2024Q1,2024-03-31,2024-04-26 18:00:00,2024-04-29 09:30:00,0.04,4.0
707,2024-08-30,2024-08-30 15:00:00,2024H1,2024-06-30,2024-08-29 18:00:00,2024-08-30 09:30:00,0.08,8.0


## 16. 课程式“最高 ROE”筛选，只做连接结果核对

下面在每天的 6 只资产中选原始累计 ROE 最高者，用来确认横截面结果可分组、可排序。它**不是策略回测**：

- 同一天同一报告期内，本模拟数据的 ROE 在发布之间不变；
- 原始 ROE 跨行业未做口径统一和行业中性化；
- 没有收益标签、执行价格、持仓期、换手、成本或样本外检验；
- “某资产入选多少天”只描述这张模拟输入表。

In [15]:
# 每个 trade_at 按 ROE 降序、symbol 升序稳定排序，取第一行。
top_roe_by_day = (
    pit_result.dropna(subset=["roe_ytd"])
    .sort_values(
        ["trade_at", "roe_ytd", "symbol"],
        ascending=[True, False, True],
    )
    .groupby("trade_at", sort=False)
    .head(1)
    .reset_index(drop=True)
)
selection_counts = (
    top_roe_by_day["symbol"]
    .value_counts()
    .rename_axis("symbol")
    .rename("selected_days")
    .to_frame()
)

assert top_roe_by_day["trade_at"].is_unique
assert len(top_roe_by_day) == pit_result["trade_at"].nunique()
print("每日横截面数 =", len(top_roe_by_day))
display(top_roe_by_day[["date", "symbol", "sector", "report_label", "roe_ytd"]].head())
selection_counts

每日横截面数 = 123


,date,symbol,sector,report_label,roe_ytd
0,2024-03-20,TECH_A,科技,2023Q3,0.12
1,2024-03-21,TECH_A,科技,2023Q3,0.12
2,2024-03-22,TECH_A,科技,2023Q3,0.12
3,2024-03-25,TECH_A,科技,2023Q3,0.12
4,2024-03-26,TECH_A,科技,2023Q3,0.12


,selected_days
symbol,
TECH_A,123


## 17. 量化意义与方法边界

- `concat` 只负责沿轴组合，不替你检查业务主键、单位、复权口径或时区。
- `merge` 应显式写 `on/how/validate`；默认 `inner` 可能静默丢失左表记录。
- `_merge` 说明键是否匹配；普通字段的 `NaN` 还可能来自源数据本身。
- 多对多连接不一定错误，但必须是明确的业务设计；事实表补维度通常不应多对多。
- `direction="backward"` 只有在右键确实是 `available_at` 时才能防未来函数；换成 `report_period` 仍会泄漏。
- `by="symbol"` 不可省略，否则一只资产可能匹配到另一只资产最近发布的财报。
- `tolerance` 是研究规则，不是“越大越好”；过大允许陈旧数据，过小会扩大缺失。
- 收盘后公告何时可用取决于时区、采集延迟和执行时点；真实系统应保存原始发布时间、入库时间和规则版本。
- 同一报告可能更正或重发；必须保存版本并明确“当时看到的版本”，不能用最终修订值覆盖历史。
- 证券代码可能复用或变更；生产连接应使用稳定证券标识并记录公司行动。
- ROE 可能是单季、累计、年化或 TTM；Q1“重置”只适用于累计口径。
- 原始 ROE 跨行业比较会受杠杆、会计口径和行业结构影响；排名本身不是收益证据。
- 本 fixture 没有停牌、退市、幸存者偏差、货币差异和财报修订，不构成真实投资研究。

## 18. 可复用结论与练习

```text
原始事实表 → 主键/单位/时间契约
           → 精确维度 merge(validate + indicator)
           → 财报 published_at / available_at
           → 按时间、资产排序
           → merge_asof(by + backward + tolerance)
           → 行数、时点、发布日期阶梯审计
```

练习：

1. 把 `ALPHA` 新报告的 `available_at` 改到 2025-01-03 14:00，先写出 15:00 快照的预期，再运行断言。
2. 删除 `by="symbol"`，找出发生跨资产误配的最小输入；不要把错误结果用于后续计算。
3. 给行业维表加入重复 `ALPHA`，比较没有 `validate` 时行数如何膨胀。
4. 给同一财报加入“初始版/修订版”，设计 `version`、`available_at` 和去重规则。
5. 把 `tolerance` 从 220 天改为 90 天，统计缺失率，并解释它是研究规则变化而不是数据自动变差。